# Lab 2: Grounding & Context Engineering (Hybrid Search to Knowledge Triples)
**Course:** Intelligent System Development (知能システム開発特論)  
**Theme:** Beyond Naive Vector Search: BM25, Reciprocal Rank Fusion (RRF), Cross-Encoder Reranking, and Knowledge Graph Triples.

### Assignment Roadmap
- **Part 1:** Schema Definitions for Graph Triples (`EntityTriple`, `ExtractionResult`)
- **Part 2 (TODO 1):** Implement BM25 Okapi Lexical Retrieval (Slide 7)
- **Part 3 (TODO 2):** Implement Reciprocal Rank Fusion (RRF) (Slide 8)
- **Part 4 (TODO 3):** Implement Cross-Encoder Style Reranker (Slide 9)
- **Part 5 (TODO 4):** Implement Rule-Based Knowledge Triple Extraction (Slide 11)
- **Part 6:** Pipeline Verification, Graph Construction & Neighborhood Traversal

Work from top to bottom. Each TODO has a test cell right below it: it prints `passed` or tells you what is still missing.

---

**日本語 / Japanese**

### 演習2:グラウンディングと文脈工学(ハイブリッド検索から知識トリプルへ)
**科目:** 知能システム開発特論  
**テーマ:** 単純なベクトル検索を超える:BM25、相互順位融合(RRF)、クロスエンコーダによる再順位付け、知識グラフのトリプル。

**課題の流れ**
- パート1:グラフ用トリプルのスキーマ定義(`EntityTriple`、`ExtractionResult`)
- パート2(TODO 1):BM25 Okapi 語彙検索の実装(スライド7)
- パート3(TODO 2):相互順位融合 RRF の実装(スライド8)
- パート4(TODO 3):クロスエンコーダ型リランカーの実装(スライド9)
- パート5(TODO 4):ルールベースの知識トリプル抽出の実装(スライド11)
- パート6:パイプラインの確認、グラフ作成、近傍探索

上から順に進めてください。各TODOの下にテスト用セルがあり、「合格」か、まだ足りない点を表示します。

## 0. Setup & Dependencies
Install the required libraries if running on Google Colab or a fresh virtual environment.

---

**日本語:** セットアップと依存ライブラリ。Google Colab や新しい仮想環境で実行する場合は、必要なライブラリをインストールします。

In [ ]:
!pip install pydantic networkx numpy pytest google-genai

In [ ]:
import json
import math
import os
import re
from typing import Any, Dict, List, Tuple, Set

import networkx as nx
import numpy as np
from pydantic import BaseModel, Field


def run_check(todo: int, test_fn) -> None:
    """Run a test function and print a friendly result instead of stopping the notebook.
    テスト関数を実行し、ノートブックを止めずに結果を表示します。"""
    try:
        test_fn()
        print(f"TODO {todo} passed successfully! / TODO {todo} 合格です!")
    except AssertionError as e:
        print(f"TODO {todo} not passed yet / まだ合格していません: {e}")
    except Exception as e:
        print(f"TODO {todo} raised an error / エラーが発生しました {type(e).__name__}: {e}")

## 1. Schema Definitions for Knowledge Triples
We model extracted relational facts using Pydantic schemas. Each relation is represented as a directed triple: `(subject) --[predicate]--> (object)`.

---

**日本語:** 知識トリプルのスキーマ定義。抽出した関係を Pydantic のスキーマで表します。各関係は有向トリプル `(主語) --[述語]--> (目的語)` です。

In [ ]:
class EntityTriple(BaseModel):
    """Structured knowledge triple representing an extracted semantic relationship."""
    subject: str = Field(description="The primary entity or actor")
    predicate: str = Field(description="The relationship or directional action")
    object: str = Field(description="The target entity, property, or concept")


class ExtractionResult(BaseModel):
    """Container for document extraction."""
    document_id: str
    triples: List[EntityTriple] = Field(default_factory=list)

## 2. In-Memory Corpus
This benchmark corpus contains passages discussing hybrid retrieval, error identifiers (e.g., `ERR-4039`), and automotive maintenance to test lexical versus semantic retrieval paths.

---

**日本語:** メモリ上のコーパス。ハイブリッド検索、エラーコード(`ERR-4039` など)、自動車整備に関する文章が入っています。語彙検索と意味検索の違いを確かめるためのものです。

In [ ]:
CORPUS: List[Dict[str, str]] = [
    {
        "id": "doc_01",
        "title": "Hybrid Search Overview",
        "content": "Hybrid search combines dense vector retrieval and sparse lexical search like BM25 to mitigate exact keyword failure in naive vector RAG."
    },
    {
        "id": "doc_02",
        "title": "Reciprocal Rank Fusion",
        "content": "Reciprocal rank fusion mitigates ranking bias by fusing sparse and dense rankings. Reciprocal rank fusion relies on reciprocal rank sums across multiple retrieved lists."
    },
    {
        "id": "doc_03",
        "title": "Database Connection Failure",
        "content": "Error code ERR-4039 indicates a severe database timeout during connection pooling. The servicing procedure relies on restarting the backend cluster."
    },
    {
        "id": "doc_04",
        "title": "Automotive Servicing Guidelines",
        "content": "Regular vehicle maintenance and automobile servicing avoids costly car repairs. Routine engine servicing relies on changing fluids every 5000 miles."
    },
    {
        "id": "doc_05",
        "title": "GraphRAG and Triples",
        "content": "GraphRAG combines knowledge graphs and dense embeddings to support multi-hop reasoning. Structured extraction relies on extracting subject predicate object triples."
    }
]

print(f"Loaded {len(CORPUS)} documents into corpus. / コーパスに {len(CORPUS)} 件の文書を読み込みました。")

## 3. Lexical Retrieval: BM25 Okapi (TODO 1)
Dense embeddings frequently struggle with exact alphanumeric codes (e.g., `ERR-4039`) or rare terms.

The BM25 Okapi score for document $D$ and query $Q$ is computed as:
$$\text{IDF}(t) = \ln\left(\frac{N - n(t) + 0.5}{n(t) + 0.5} + 1\right)$$
$$\text{Score}(D, Q) = \sum_{t \in Q} \text{IDF}(t) \cdot \frac{\text{TF}(t, D) \cdot (k_1 + 1)}{\text{TF}(t, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Complete the `score` method below.

---

**日本語:** 語彙検索 BM25 Okapi(TODO 1)。密ベクトル検索は、`ERR-4039` のような英数字コードや珍しい語が苦手です。BM25 は、検索語が文書に何回出るか(TF)、その語がどれだけ珍しいか(IDF)、文書の長さを使ってスコアを計算します。式は上のとおりです。下の `score` メソッドを完成させてください。

In [ ]:
class BM25Retriever:
    """In-memory BM25 Okapi lexical retriever."""

    def __init__(self, corpus: List[Dict[str, str]], k1: float = 1.5, b: float = 0.75):
        self.corpus = corpus
        self.k1 = k1
        self.b = b
        self.doc_len: Dict[str, int] = {}
        self.avg_doc_len: float = 0.0
        self.doc_freqs: Dict[str, int] = {}
        self.term_freqs: Dict[str, Dict[str, int]] = {}
        self.num_docs = len(corpus)
        self._build_index()

    def _tokenize(self, text: str) -> List[str]:
        return [w.lower().strip(".,!?:;\"'()[]{}") for w in text.split() if w.strip(".,!?:;\"'()[]{}")]

    def _build_index(self):
        total_len = 0
        for doc in self.corpus:
            doc_id = doc["id"]
            tokens = self._tokenize(doc["content"])
            self.doc_len[doc_id] = len(tokens)
            total_len += len(tokens)
            tf: Dict[str, int] = {}
            for token in tokens:
                tf[token] = tf.get(token, 0) + 1
            self.term_freqs[doc_id] = tf
            for token in set(tokens):
                self.doc_freqs[token] = self.doc_freqs.get(token, 0) + 1
        self.avg_doc_len = total_len / self.num_docs if self.num_docs > 0 else 1.0

    def score(self, query: str) -> List[Tuple[str, float]]:
        """
        TODO 1: Return [(doc_id, bm25_score), ...] for EVERY document, sorted best first.

        Hints: use self.doc_freqs, self.term_freqs, self.doc_len, self.avg_doc_len.
               Skip query terms that are not in the index. Guard against division by zero.

        TODO 1: 全文書について [(doc_id, bm25_score), ...] を、スコアの高い順に返す。
        ヒント:self.doc_freqs, self.term_freqs, self.doc_len, self.avg_doc_len を使う。
               索引にない検索語は飛ばす。0 での割り算に注意する。
        """
        query_tokens = self._tokenize(query)
        scores: List[Tuple[str, float]] = []

        for doc in self.corpus:
            scores.append((doc["id"], 0.0))   # placeholder: replace with real BM25 scores
        return scores

### Test TODO 1 / TODO 1 のテスト

In [ ]:
bm25 = BM25Retriever(CORPUS)
test_scores = bm25.score("ERR-4039")
print("BM25 Scores for 'ERR-4039':", test_scores)

def test_todo1():
    assert test_scores[0][0] == "doc_03", "doc_03 must be top-ranked for ERR-4039! / ERR-4039 では doc_03 が1位になるはずです"
    assert test_scores[0][1] > 0.0, "Top document must have a positive BM25 score! / 1位の文書のスコアは正の値になるはずです"

run_check(1, test_todo1)

## 4. Reciprocal Rank Fusion (TODO 2)
RRF merges rankings from dense and sparse retrieval without requiring uncalibrated score normalization:

$$\text{RRF\_Score}(d \in D) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
where $r_m(d)$ is the 1-based rank position of document $d$ in system $m$, and $k$ (default 60) dampens top-rank disparity.

---

**日本語:** 相互順位融合 RRF(TODO 2)。RRF は、スコアではなく順位だけを使って密検索と疎検索の結果を統合します。スコアの尺度をそろえる必要がありません。式の $r_m(d)$ は、検索システム $m$ での文書 $d$ の順位(1から数えます)、$k$(標準は60)は上位順位の差をやわらげる定数です。

In [ ]:
def reciprocal_rank_fusion(
    dense_ranked_ids: List[str],
    sparse_ranked_ids: List[str],
    k: int = 60,
) -> List[Tuple[str, float]]:
    """
    TODO 2: Merge two ranked lists of doc IDs. Return [(doc_id, rrf_score), ...] best first.
    A document missing from one list gets no contribution from that list.

    TODO 2: 2つの順位リストを統合し、[(doc_id, rrf_score), ...] を高い順に返す。
    片方のリストにない文書は、そのリストからの加点がない。順位は1から数える。
    """
    scores: Dict[str, float] = {}

    for doc_id in dense_ranked_ids + sparse_ranked_ids:
        scores[doc_id] = 0.0                 # placeholder: replace with RRF
    return list(scores.items())

### Test TODO 2 (Slide 8 table) / TODO 2 のテスト(スライド8の表)

In [ ]:
# Slide 8 example:
# Doc A: Dense rank 1, Sparse rank 3 -> 1/61 + 1/63 = 0.0323
# Doc B: Dense rank 2, Sparse rank 1 -> 1/62 + 1/61 = 0.0325   (winner)
# Doc C: Dense rank 3, not in sparse -> 1/63            = 0.0159
test_rrf = reciprocal_rank_fusion(["A", "B", "C"], ["B", "X", "A"], k=60)
print("RRF results:", [(d, round(s, 4)) for d, s in test_rrf])

def test_todo2():
    res = dict(test_rrf)
    assert len(res) == 4, "Every document must appear once. / すべての文書が1回ずつ現れるはずです"
    assert abs(res["A"] - (1/61 + 1/63)) < 1e-6, "Doc A should be about 0.0323 / 文書Aは約0.0323になるはずです"
    assert abs(res["B"] - (1/62 + 1/61)) < 1e-6, "Doc B should be about 0.0325 / 文書Bは約0.0325になるはずです"
    assert abs(res["C"] - 1/63) < 1e-6, "Doc C should be about 0.0159 / 文書Cは約0.0159になるはずです"
    assert test_rrf[0][0] == "B", "Doc B should rank first (good in both lists). / 文書Bが1位になるはずです(両方のリストで上位)"

run_check(2, test_todo2)

## 5. Cross-Encoder Style Reranker (TODO 3)
Rerankers jointly process the query and document tokens to eliminate false positives from bi-encoder retrieval. Here we imitate a cross-encoder with a token-overlap score (the helper `_norm_tokens` is provided).

---

**日本語:** クロスエンコーダ型リランカー(TODO 3)。リランカーは、質問と文書の単語をいっしょに処理して、第1段階の検索の誤りを取り除きます。ここでは単語の重なりでクロスエンコーダをまねます(補助関数 `_norm_tokens` は用意済みです)。

In [ ]:
STOPWORDS = {"a", "an", "the", "of", "in", "on", "to", "and", "or", "with", "how", "does", "do", "is", "are", "for", "by"}

def _norm_tokens(text: str) -> Set[str]:
    """Lowercase, remove punctuation, drop stopwords, crude plural/verb-ending stemming."""
    out = set()
    for w in re.findall(r"[a-z0-9][a-z0-9\-]*", text.lower()):
        if w in STOPWORDS:
            continue
        for suf in ("es", "s"):
            if w.endswith(suf) and len(w) > 4:
                w = w[:-len(suf)]
                break
        if w.endswith("e") and len(w) > 4:
            w = w[:-1]
        out.add(w)
    return out


def cross_encoder_rerank(
    query: str,
    candidate_docs: List[Dict[str, str]],
    top_k: int = 3,
) -> List[Dict[str, Any]]:
    """
    TODO 3: Score each candidate AGAINST THE QUERY and return the best top_k as dicts with keys:
            id, title, content, rerank_score (rounded to 4 decimals), sorted best first.

    Score = (number of shared tokens) / (number of query tokens)   [use _norm_tokens on both]
            + 0.1 bonus if the whole query string appears inside the passage.

    TODO 3: 各候補を「質問と照らして」採点し、上位 top_k 件を返す。
    キーは id, title, content, rerank_score(小数4桁)、スコアの高い順。
    スコア = (共通トークン数) / (質問のトークン数)。両方に _norm_tokens を使う。
             質問の文字列全体が文章に含まれていれば +0.1。
    """
    scored: List[Dict[str, Any]] = []

    for doc in candidate_docs:
        scored.append({"id": doc["id"], "title": doc.get("title", ""), "content": doc["content"],
                       "rerank_score": 0.0})   # placeholder: compute a real score
    return scored[:top_k]

### Test TODO 3 / TODO 3 のテスト

In [ ]:
test_rerank = cross_encoder_rerank("hybrid search", CORPUS, top_k=2)
print("Reranked top candidates:", [(d["id"], d["rerank_score"]) for d in test_rerank])

def test_todo3():
    assert len(test_rerank) == 2, "Return exactly top_k documents. / ちょうど top_k 件を返してください"
    assert test_rerank[0]["id"] == "doc_01", "doc_01 should rank highest for 'hybrid search'! / 'hybrid search' では doc_01 が1位になるはずです"
    assert test_rerank[0]["rerank_score"] > 0, "The best document needs a positive score. / 1位の文書のスコアは正の値になるはずです"

run_check(3, test_todo3)

## 6. Knowledge Triple Extraction (TODO 4)
Extract relational facts to construct a local knowledge graph. The fixed list `RELATIONS` is our **schema**: only these relation types are allowed.

---

**日本語:** 知識トリプルの抽出(TODO 4)。文書から関係を取り出して、小さな知識グラフを作ります。固定リスト `RELATIONS` が**スキーマ**で、この種類の関係だけが許されます。

In [ ]:
RELATIONS = {" mitigates ": "mitigates", " relies on ": "relies_on", " combines ": "combines"}

def extract_triples_rule_based(document: Dict[str, str]) -> ExtractionResult:
    """
    TODO 4: Split the document into sentences and extract (subject, predicate, object) triples
            as EntityTriple objects based on the relation anchors in RELATIONS.
            Strip whitespace and use the predicate name from RELATIONS (e.g. "relies_on").

    TODO 4: 文書を文に分け、RELATIONS のキーワードを手がかりに
    (主語, 述語, 目的語) を EntityTriple として取り出す。
    前後の空白を取り、述語は RELATIONS の名前(例:"relies_on")を使う。
    """
    triples: List[EntityTriple] = []

    return ExtractionResult(document_id=document["id"], triples=triples)

### Test TODO 4 / TODO 4 のテスト

In [ ]:
res = extract_triples_rule_based(CORPUS[1])
print("Extracted triples from doc_02:")
for t in res.triples:
    print(f"  ({t.subject}) --[{t.predicate}]--> ({t.object})")

def test_todo4():
    assert len(res.triples) >= 2, "Should extract at least two triples from doc_02! / doc_02 から2つ以上のトリプルを取り出してください"
    assert {t.predicate for t in res.triples} <= set(RELATIONS.values()), "Use the predicate names from RELATIONS. / 述語は RELATIONS の名前を使ってください"

run_check(4, test_todo4)

## 7. Pipeline Integration & NetworkX Graph Traversal
We now run the toy dense retriever, fuse the sparse and dense lists, and construct the GraphRAG relational graph. The helpers below are provided: do not edit them.

The toy dense retriever puts synonyms on the same axis (`car` = `automobile`) and ignores unknown tokens such as `ERR-4039`. This imitates the strengths and weaknesses of real embeddings.

---

**日本語:** パイプラインの統合と NetworkX によるグラフ探索。ここまでの部品を組み合わせ、疎検索と密検索を RRF で統合し、GraphRAG 用の関係グラフを作ります。下の補助関数は用意済みなので、編集しないでください。簡易の密検索は、同義語を同じ軸に置き(`car` = `automobile`)、`ERR-4039` のような未知の語は無視します。実際の埋め込みモデルの長所と短所をまねたものです。

In [ ]:
CONCEPTS = [
    {"car", "cars", "automobile", "vehicle"},
    {"repair", "repairs", "maintenance", "fix", "servicing", "service"},
    {"search", "retrieval", "retrieve", "lookup", "query", "queries"},
    {"hybrid", "fusion", "combine", "combines", "blend"},
    {"embedding", "embeddings", "vector", "vectors", "semantic", "dense"},
    {"keyword", "keywords", "lexical", "bm25", "sparse", "exact", "identifiers"},
    {"rerank", "reranking", "reranker", "cross-encoder", "bi-encoder"},
    {"hallucination", "hallucinations", "poisoning", "noise", "irrelevant", "failure", "failures", "mismatch"},
    {"graph", "graphrag", "knowledge", "triples", "triple", "multi-hop", "entity"},
    {"summarize", "summarization", "summary", "themes", "corpus", "corpora"},
    {"chunk", "chunks", "chunking", "windowing", "split", "splits", "paragraph", "paragraphs"},
    {"database", "timeout", "connection", "error"},
]

def toy_embed(text: str) -> np.ndarray:
    vec = np.zeros(len(CONCEPTS))
    for w in re.findall(r"[a-z0-9][a-z0-9\-]*", text.lower()):
        for i, group in enumerate(CONCEPTS):
            if w in group:
                vec[i] += 1.0
    return vec

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na > 0 and nb > 0 else 0.0

def dense_rank(query: str, corpus: List[Dict[str, str]]) -> List[str]:
    q = toy_embed(query)
    scored = [(d["id"], cosine_similarity(q, toy_embed(d["content"]))) for d in corpus]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, s in scored if s > 0]   # zero similarity = not retrieved

def build_graph(results: List[ExtractionResult]) -> nx.DiGraph:
    G = nx.DiGraph()
    for result in results:
        for triple in result.triples:
            G.add_edge(triple.subject, triple.object, relation=triple.predicate)
    return G

def run_query(query: str, corpus: List[Dict[str, str]], bm25: BM25Retriever) -> None:
    print(f"\n=== Query: '{query}' ===")
    sparse_ids = [doc_id for doc_id, s in bm25.score(query) if s > 0]   # zero score = not retrieved
    dense_ids = dense_rank(query, corpus)
    print(f"  BM25  Top-3 : {sparse_ids[:3]}")
    print(f"  Dense Top-3 : {dense_ids[:3]}")
    fused = reciprocal_rank_fusion(dense_ids, sparse_ids, k=60)
    print("  RRF merged  : " + ", ".join(f"{d} ({s:.5f})" for d, s in fused[:3]))
    keep = {d for d, _ in fused[:5]}
    final = cross_encoder_rerank(query, [d for d in corpus if d["id"] in keep], top_k=2)
    for p in final:
        if p['rerank_score'] <= 0:
            continue   # do not show passages the reranker scored as irrelevant
        print(f"  Rerank [{p['id']}] score={p['rerank_score']}: {p['content'][:70]}...")

### Run Benchmark Queries & Inspect Knowledge Graph / ベンチマーク検索の実行と知識グラフの確認

In [ ]:
bm25 = BM25Retriever(CORPUS)

# 1. Hybrid query with semantic + exact terms
run_query("How does hybrid search mitigate exact keyword failure in naive vector RAG?", CORPUS, bm25)

# 2. Exact identifier query (sparse BM25 dominance)
run_query("ERR-4039", CORPUS, bm25)

# 3. Paraphrase query (dense dominance)
run_query("car repair tips", CORPUS, bm25)

# 4. Build Knowledge Graph from all documents
print("\n--- Building Knowledge Graph ---")
all_extractions = [extract_triples_rule_based(d) for d in CORPUS]
G = build_graph(all_extractions)
print(f"Graph constructed: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

for u, v, data in G.edges(data=True):
    print(f"  ({u}) --[{data['relation']}]--> ({v})")

start_node = "Reciprocal rank fusion"
if start_node in G:
    two_hop = sorted(nx.single_source_shortest_path_length(G, start_node, cutoff=2))
    print(f"\nTwo-hop neighborhood of '{start_node}':\n  {two_hop}")